In [ ]:
"""
Final XGBoost model:
direct training/test data input, preprocessing, cross-validation,
model evaluation, result export, and final model saving.
"""

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

In [ ]:
# 1. File paths
# ============================================================
TRAIN_FILE = Path(r"D:\training_set.xlsx")
TEST_FILE = Path(r"D:\test_set.xlsx")

OUTPUT_DIR = Path(r"D:\XGBoost_final_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. Input features and target
# ============================================================
NUMERIC_FEATURES = [
    "Bulk density",
    "Initial pH",
    "Initial C/N ratio",
    "Initial moisture content",
    "Composting duration",
    "Reactor volume",
]

CATEGORICAL_FEATURES = [
    "Composting type",
    "Bulking agent type",
    "Treatment condition",
    "Turning regime",
    "Aeration regime",
]

TARGET = "Score"


In [ ]:
# 3. Read the predefined training and test datasets
# ============================================================
train_df = pd.read_excel(TRAIN_FILE)
test_df = pd.read_excel(TEST_FILE)

required_columns = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
    + [TARGET]
)

missing_train_columns = [
    c for c in required_columns if c not in train_df.columns
]
missing_test_columns = [
    c for c in required_columns if c not in test_df.columns
]

if missing_train_columns:
    raise ValueError(
        f"Training dataset is missing the following columns: "
        f"{missing_train_columns}"
    )

if missing_test_columns:
    raise ValueError(
        f"Test dataset is missing the following columns: "
        f"{missing_test_columns}"
    )

# Do not perform additional imputation or remove samples.
# Stop immediately if missing values are detected.
if train_df[required_columns].isna().sum().sum() > 0:
    raise ValueError(
        "Missing values were detected in the training dataset. "
        "Please confirm that the finalized dataset is being used."
    )

if test_df[required_columns].isna().sum().sum() > 0:
    raise ValueError(
        "Missing values were detected in the test dataset. "
        "Please confirm that the finalized dataset is being used."
    )

print("Training samples:", len(train_df))
print("Test samples:    ", len(test_df))


In [ ]:
# 4. Build X and y
# ============================================================
X_train = train_df[
    NUMERIC_FEATURES + CATEGORICAL_FEATURES
].copy()

X_test = test_df[
    NUMERIC_FEATURES + CATEGORICAL_FEATURES
].copy()

y_train = train_df[TARGET].astype(float).to_numpy()
y_test = test_df[TARGET].astype(float).to_numpy()

# The original literature/experiment identifiers were removed from the
# direct modeling datasets. Deterministic row-based sample IDs are created
# only to preserve the original prediction-output structure.
id_train = pd.Series(
    [f"Training_{i:04d}" for i in range(1, len(train_df) + 1)]
)

id_test = pd.Series(
    [f"Test_{i:04d}" for i in range(1, len(test_df) + 1)]
)


In [ ]:
# 5. Data preprocessing
# ============================================================
# Continuous variables: Z-score standardization
# Categorical variables: One-Hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)


In [ ]:
# 6. Final XGBoost hyperparameters
# ============================================================
xgb_model = XGBRegressor(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.07,
    subsample=0.90,
    colsample_bytree=1.00,
    min_child_weight=3,
    gamma=0,
    reg_alpha=0,
    reg_lambda=3,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)


In [ ]:
# 7. Build the complete pipeline
# ============================================================
model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("xgboost", xgb_model),
    ]
)


In [ ]:
# 8. Five-fold cross-validation on the training dataset
# ============================================================
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# OOF prediction:
# each training sample is predicted by a model that was not trained on it.
oof_pred = cross_val_predict(
    model,
    X_train,
    y_train,
    cv=cv,
    n_jobs=1,
)


In [ ]:
# 9. Fit the final model using the complete training dataset
# ============================================================
model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)


In [ ]:
# 10. Model evaluation
# ============================================================
def evaluate(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae


train_r2, train_rmse, train_mae = evaluate(
    y_train, train_pred
)

oof_r2, oof_rmse, oof_mae = evaluate(
    y_train, oof_pred
)

test_r2, test_rmse, test_mae = evaluate(
    y_test, test_pred
)

print("\n================ Final Model Performance ================")

print("\nTraining fitted")
print(f"R²   = {train_r2:.4f}")
print(f"RMSE = {train_rmse:.4f}")
print(f"MAE  = {train_mae:.4f}")

print("\nTraining 5-fold OOF")
print(f"R²   = {oof_r2:.4f}")
print(f"RMSE = {oof_rmse:.4f}")
print(f"MAE  = {oof_mae:.4f}")

print("\nTest")
print(f"R²   = {test_r2:.4f}")
print(f"RMSE = {test_rmse:.4f}")
print(f"MAE  = {test_mae:.4f}")


In [ ]:
# 11. Export predictions
# ============================================================
train_output = pd.DataFrame({
    "Sample_ID": id_train,
    "Observed": y_train,
    "Training_Fitted_Predicted": train_pred,
    "Training_OOF_Predicted": oof_pred,
})

test_output = pd.DataFrame({
    "Sample_ID": id_test,
    "Observed": y_test,
    "Test_Predicted": test_pred,
})

performance_output = pd.DataFrame({
    "Dataset": [
        "Training_fitted",
        "Training_5fold_OOF",
        "Test",
    ],
    "R2": [
        train_r2,
        oof_r2,
        test_r2,
    ],
    "RMSE": [
        train_rmse,
        oof_rmse,
        test_rmse,
    ],
    "MAE": [
        train_mae,
        oof_mae,
        test_mae,
    ],
})


In [ ]:
# 12. Save Excel results
# ============================================================
excel_file = OUTPUT_DIR / "XGBoost_final_results.xlsx"

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    performance_output.to_excel(
        writer,
        sheet_name="Model_Performance",
        index=False,
    )

    train_output.to_excel(
        writer,
        sheet_name="Training_Predictions",
        index=False,
    )

    test_output.to_excel(
        writer,
        sheet_name="Test_Predictions",
        index=False,
    )

print("\nResults saved to:", excel_file)


In [ ]:
# ============================================================
# 13. Save the final model
# ============================================================
model_file = OUTPUT_DIR / "XGBoost_final_model.joblib"
joblib.dump(model, model_file)

print("Final model saved to:", model_file)